In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sparrow import Protein
from tqdm import tqdm

plt.rcParams['svg.fonttype'] = 'none'

In [2]:
WTCRX = 'MMAYMNPGPHYSVNALALSGPSVDLMHQAVPYPSAPRKQRRERTTFTRSQLEELEALFAKTQYPDVYAREEVALKINLPESRVQVWFKNRRAKCRQQRQQQKQQQQPPGGQAKARPAKRKAGTSPRPSTDVCPDPLGISDSYSPPLPGPSGSPTTAVATVSIWSPASESPLPEAQRAGLVASGPSLTSAPYAMTYAPASAFCSSPSAYGSPSSYFSGLDPYLSPMVPQLGGPALSPLSGPSVGPSLAQSPTSLSGQSYGAYSPVDSLEFKDPTGTWKFTYNPMDPLDYKDQSAWKFQIL'

In [3]:
AAS = list('ACDEFGHIKLMNPQRSTVWY')

rows = []
for pos, wt_aa in enumerate(WTCRX):
    for mut_aa in AAS:
        if mut_aa == wt_aa:
            continue
        mutation = f'{wt_aa}{pos + 1}{mut_aa}'   # 1-indexed
        sequence = WTCRX[:pos] + mut_aa + WTCRX[pos + 1:]
        rows.append({'mutation': mutation, 'sequence': sequence})
df_mutations = pd.DataFrame(rows)

In [4]:
CATEGORIES = ['CLV', 'DEG', 'DOC', 'LIG', 'MOD', 'TRG']

def get_elm_set(sequence):
    """Return set of unique ELM identifiers with position appended: IDENTIFIER_start-end"""
    p = Protein(sequence)
    return set(f"{elm.identifier}_{elm.start}-{elm.end}" for elm in p.elms)

def count_by_category(elm_set):
    counts = {cat: 0 for cat in CATEGORIES}
    for uid in elm_set:
        cat = uid.split('_')[0]
        if cat in counts:
            counts[cat] += 1
    return counts

def locations_by_category(elm_set):
    """Return dict of category -> sorted list of 'start-end' strings"""
    locs = {cat: [] for cat in CATEGORIES}
    for uid in elm_set:
        cat = uid.split('_')[0]
        if cat in locs:
            pos = uid.rsplit('_', 1)[-1]   # last element is 'start-end'
            locs[cat].append(pos)
    return {cat: sorted(v) for cat, v in locs.items()}

# --- WT reference ---
wt_elms = get_elm_set(WTCRX)
wt_counts = count_by_category(wt_elms)

# --- Mutant loop ---
rows = []
for _, row in tqdm(df_mutations.iterrows(), total=len(df_mutations)):
    mut_elms = get_elm_set(row['sequence'])
    mut_counts = count_by_category(mut_elms)
    mut_locs   = locations_by_category(mut_elms)

    added   = mut_elms - wt_elms
    removed = wt_elms - mut_elms

    r = {'mutation': row['mutation']}

    for cat in CATEGORIES:
        r[f'{cat}_delta']     = abs(mut_counts[cat] - wt_counts[cat])
        r[f'{cat}_added']     = sum(1 for e in added   if e.split('_')[0] == cat)
        r[f'{cat}_removed']   = sum(1 for e in removed if e.split('_')[0] == cat)
        r[f'{cat}_locations'] = mut_locs[cat]

    r['SLiM_delta']   = abs(len(mut_elms) - len(wt_elms))
    r['SLiM_added']   = len(added)
    r['SLiM_removed'] = len(removed)
    r['which_added']   = sorted(added)
    r['which_removed'] = sorted(removed)

    rows.append(r)

df_elm_mutations = pd.DataFrame(rows)

100%|██████████| 5681/5681 [00:42<00:00, 132.20it/s]


In [5]:
df_elm_mutations = df_mutations.merge(df_elm_mutations, on='mutation')

# move 'sequence' to last column
cols = [c for c in df_elm_mutations.columns if c != 'sequence'] + ['sequence']
df_elm_mutations = df_elm_mutations[cols]

In [6]:
# Build WT row
wt_locs = locations_by_category(wt_elms)
wt_row = {'mutation': 'WT'}

for cat in CATEGORIES:
    wt_row[f'{cat}_delta']     = 0
    wt_row[f'{cat}_added']     = 0
    wt_row[f'{cat}_removed']   = 0
    wt_row[f'{cat}_locations'] = wt_locs[cat]

wt_row['SLiM_delta']   = 0
wt_row['SLiM_added']   = 0
wt_row['SLiM_removed'] = 0
wt_row['which_added']   = []
wt_row['which_removed'] = []
wt_row['sequence'] = WTCRX

df_elm_mutations = pd.concat(
    [pd.DataFrame([wt_row]), df_elm_mutations],
    ignore_index=True
)

In [7]:
df_elm_mutations.to_csv('../../Data/ELM_mutations.csv', index=False)